# 6. MMP Fragment Reassembly

## 이번 노트북에서 할 것
- core의 attachment point(`[*:1]`)에 `replacement_library.py`의 candidate SMILES를 이어붙여 새 분자 재조립
- `Chem.molzip()` (또는 대안 방식) 조사 및 적용
- 재조립된 분자가 화학적으로 유효한지(`Chem.MolFromSmiles()`) 검증
- (시간 되면) 재조립 분자를 baseline classifier + toxicophore_detector로 재평가

## 간략한 정리 (05까지)
- `rdMMPA.FragmentMol()`은 분해(fragmentation)만 함, 교체(replacement)는 안 함
- `maxCuts=1`: core 없음 (2조각뿐이라 애매함) / `maxCuts=2`: core 생김 (가운데 남는 부분)
- core 있는 조합만 필터링: `[(core, chain) for core, chain in fragments if core]`
- 도구 3종 완성: `data_prep.py`(SMILES 포함) / `toxicophore_detector.py`(PAINS+BRENK) / `replacement_library.py`(치환 후보 5종)
- 미해결: 원본 SMILES + core + chain을 함께 기록하는 구조 아직 없음

## 다음에 해야 할 것
- 재조립 함수 완성 후 `src/tools/`에 저장
- 원본 분자 정보까지 함께 추적하는 기록 구조 설계
- 재조립 → 재평가까지 되면, 에이전트(LLM 판단) 계층 설계로 이동

In [18]:
!pip install rdkit -q

In [19]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

fatal: destination path 'laidd-2026' already exists and is not an empty directory.
/content/laidd-2026
/content/laidd-2026


In [20]:
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [21]:
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates

data = load_tox21_clean()
print("도구 로드 확인 완료, smiles_train 포함 여부:", 'smiles_train' in data)

[14:43:32] WARNING: not removing hydrogen atom without neighbors
[14:43:32] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:43:32] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:43:32] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:43:33] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:43:33] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:43:33] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:43:33] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:43:33] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[14:43:33] WARNING: not removing hydrogen atom without neighbors


도구 로드 확인 완료, smiles_train 포함 여부: True


In [22]:
sample_smiles = None
for s in data['smiles_train']:
    m = Chem.MolFromSmiles(s)
    if m and 15 <= m.GetNumAtoms() <= 30:
        sample_smiles = s
        break

print("선택된 분자:", sample_smiles)

선택된 분자: O=C1CC[C@@H](C(=O)N[C@H]2C[C@@H]2c2ccccc2)N1


In [23]:
def get_single_attachment_fragments(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    results = []
    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) == 2:
            results.append((parts[0], parts[1]))
    return results

pairs = get_single_attachment_fragments(sample_smiles)
for a, b in pairs:
    print(f"조각1: {a}  |  조각2: {b}")

조각1: O=C(N[C@H]1C[C@@H]1c1ccccc1)[*:1]  |  조각2: O=C1CC[C@@H]([*:1])N1
조각1: O=C1CC[C@@H](C(=O)N[*:1])N1  |  조각2: c1ccc([C@H]2C[C@@H]2[*:1])cc1
조각1: O=C1CC[C@@H](C(=O)N[C@H]2C[C@@H]2[*:1])N1  |  조각2: c1ccc([*:1])cc1


In [24]:
tox_result = detect_toxicophores(sample_smiles)
print("탐지된 문제:", tox_result)

탐지된 문제: []


In [25]:
# nitro_group이 걸리는 분자를 Tox21 안에서 찾기
test_mol_smiles = None
for s in data['smiles_train'][:500]:  # 처음 500개만 빠르게 탐색
    result = detect_toxicophores(s)
    if any(r['rule_name'] == 'nitro_group' for r in result):
        test_mol_smiles = s
        break

print("니트로기 포함 분자:", test_mol_smiles)

니트로기 포함 분자: O=[N+]([O-])c1ccc(C=NO)o1


In [26]:
# 1단계: 이 분자에서 문제구조 탐지
tox_result = detect_toxicophores(test_mol_smiles)
print("탐지된 문제:")
for r in tox_result:
    print(" ", r)

# 2단계: 분자를 두 조각으로 나누는 모든 방법 얻기
pairs2 = get_single_attachment_fragments(test_mol_smiles)
print("\n분해된 조각 쌍들:")
for a, b in pairs2:
    print(f"조각1: {a}  |  조각2: {b}")

탐지된 문제:
  {'rule_name': 'imine_1', 'atom_indices': [7, 8]}
  {'rule_name': 'nitro_group', 'atom_indices': [0, 1, 2]}
  {'rule_name': 'oxime_1', 'atom_indices': [7, 8, 9]}
  {'rule_name': 'Oxygen-nitrogen_single_bond', 'atom_indices': [1, 2]}

분해된 조각 쌍들:
조각1: O=[N+]([O-])[*:1]  |  조각2: ON=Cc1ccc([*:1])o1
조각1: O=[N+]([O-])c1ccc([*:1])o1  |  조각2: ON=C[*:1]


In [27]:
from rdkit.Chem import rdMMPA

def find_core_and_target(smiles, rule_name):
    """rule_name에 해당하는 SMARTS를 포함하는 조각을 찾아 target으로,
    나머지를 core로 반환."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    mol = Chem.MolFromSmiles(smiles)
    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', '[H]'))
            if part_mol and part_mol.HasSubstructMatch(problem_pattern):
                target = part
                new_core = parts[1 - i]
                return {"core": new_core, "target_removed": target}
    return None

result = find_core_and_target(test_mol_smiles, "nitro_group")
print(result)

{'core': 'ON=Cc1ccc([*:1])o1', 'target_removed': 'O=[N+]([O-])[*:1]'}


In [28]:
def reassemble_molecule(core_smiles, rule_name, candidate_idx=0):
    """core의 [*:1] 자리에 replacement_library의 candidate를 붙여 새 분자를 완성."""
    info = get_replacement_candidates(rule_name)
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")

    combined = Chem.molzip(core_mol, replacement_mol)
    new_smiles = Chem.MolToSmiles(combined)

    # 유효성 검증
    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }

# 아까 얻은 core로 테스트, nitro_group의 첫 번째 후보(primary amine)로 교체
result2 = reassemble_molecule("ON=Cc1ccc([*:1])o1", "nitro_group", candidate_idx=0)
print(result2)

{'new_smiles': 'Nc1ccc(C=NO)o1', 'candidate_used': 'primary amine', 'rationale': '극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함', 'is_valid': True}


In [29]:
%%writefile src/tools/molecule_editor.py
from rdkit import Chem
from rdkit.Chem import rdMMPA
from src.tools.replacement_library import get_replacement_candidates

def find_core_and_target(smiles: str, rule_name: str):
    """분자에서 rule_name에 해당하는 문제구조를 담은 조각(target)과
    나머지 뼈대(core)를 찾아서 반환. 못 찾으면 None."""
    info = get_replacement_candidates(rule_name)
    if info is None:
        return None

    problem_pattern = Chem.MolFromSmarts(info['problem_smarts'])
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fragments = rdMMPA.FragmentMol(mol, maxCuts=1, resultsAsMols=False)

    for core, chain in fragments:
        if core:
            continue
        parts = chain.split('.')
        if len(parts) != 2:
            continue
        for i, part in enumerate(parts):
            part_mol = Chem.MolFromSmiles(part.replace('[*:1]', '[H]'))
            if part_mol and part_mol.HasSubstructMatch(problem_pattern):
                return {"core": parts[1 - i], "target_removed": part}
    return None


def reassemble_molecule(core_smiles: str, rule_name: str, candidate_idx: int = 0):
    """core의 [*:1] 자리에 replacement_library의 candidate를 붙여 새 분자를 완성."""
    info = get_replacement_candidates(rule_name)
    if info is None or candidate_idx >= len(info['candidates']):
        return None
    candidate = info['candidates'][candidate_idx]

    core_mol = Chem.MolFromSmiles(core_smiles)
    replacement_mol = Chem.MolFromSmiles(f"[*:1]{candidate['smiles']}")
    if core_mol is None or replacement_mol is None:
        return None

    try:
        combined = Chem.molzip(core_mol, replacement_mol)
        new_smiles = Chem.MolToSmiles(combined)
    except Exception:
        return None

    is_valid = Chem.MolFromSmiles(new_smiles) is not None

    return {
        "new_smiles": new_smiles,
        "candidate_used": candidate['name'],
        "rationale": candidate['rationale'],
        "is_valid": is_valid,
    }


def propose_fix(smiles: str, rule_name: str, candidate_idx: int = 0):
    """전체 파이프라인: 문제구조 위치 찾기 -> 치환 후보로 재조립까지 한번에 실행."""
    located = find_core_and_target(smiles, rule_name)
    if located is None:
        return None
    return reassemble_molecule(located['core'], rule_name, candidate_idx)

Writing src/tools/molecule_editor.py


In [30]:
!pwd
!ls src/tools/

/content/laidd-2026
data_prep.py	    __pycache__		    toxicophore_detector.py
molecule_editor.py  replacement_library.py


In [31]:
import importlib
import src.tools.molecule_editor
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import propose_fix

test = propose_fix(test_mol_smiles, "nitro_group", candidate_idx=0)
print(test)

{'new_smiles': 'Nc1ccc(C=NO)o1', 'candidate_used': 'primary amine', 'rationale': '극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함', 'is_valid': True}


In [32]:
!git add src/tools/molecule_editor.py
!git commit -m "Add molecule_editor: locate toxicophore + reassemble with replacement candidate"
!git push https://{token}@github.com/Dec32th/laidd-2026.git

[main 3b484a5] Add molecule_editor: locate toxicophore + reassemble with replacement candidate
 1 file changed, 65 insertions(+)
 create mode 100644 src/tools/molecule_editor.py
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.40 KiB | 1.40 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Dec32th/laidd-2026.git
   ff84422..3b484a5  main -> main
